# Model Training
## Health Insurance Fraud Detection System

**Purpose:** Train and evaluate four unsupervised anomaly detection models:
1. Isolation Forest
2. One-Class SVM
3. Local Outlier Factor (LOF)
4. Autoencoder (using Keras)

**Ensemble:** Combine all four models using score averaging.

---

## 1. Setup and Imports


In [6]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn models
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, 
    roc_auc_score, average_precision_score, roc_curve
)

# Keras
import keras
from keras import layers, models

# Utilities
import pickle
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✅ All imports successful!")

✅ All imports successful!


## 2. Load Preprocessed Data

In [7]:
# Load processed data from the EDA notebook
X_train = np.load('../data/processed/X_train.npy')
X_test = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test = np.load('../data/processed/y_test.npy')

# Load feature names
with open('../data/processed/feature_names.txt', 'r') as f:
    feature_names = f.read().splitlines()

# Load scaler if needed
with open('../data/processed/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

print(f"Training set: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
print(f"Test set: {X_test.shape[0]:,} samples, {X_test.shape[1]} features")
print(f"Training anomaly rate: {y_train.mean():.2%}")
print(f"Test anomaly rate: {y_test.mean():.2%}")
print(f"Features: {feature_names}")

Training set: 8,000 samples, 10 features
Test set: 2,000 samples, 10 features
Training anomaly rate: 3.94%
Test anomaly rate: 3.95%
Features: ['cpt_procedure_code', 'billed_amount', 'amount_bucket', 'year', 'month', 'day_of_week', 'day_of_month', 'quarter', 'icd10_diagnosis_code_freq', 'claim_status_freq']


## 3. Evaluation Function

Define a function to evaluate anomaly detection models using:
- **Precision@100**: Primary metric - proportion of true anomalies in top 100
- **AUC-ROC**: Threshold-independent holistic measure
- **AUC-PR**: Better for imbalanced data
- **F1-Score**: Balance between precision and recall

In [8]:
def evaluate_model(y_true, scores, k=100):
    """
    Evaluate anomaly detection model performance.
    
    Parameters:
    -----------
    y_true : array
        True labels (0 = normal, 1 = anomaly)
    scores : array
        Anomaly scores (higher = more anomalous)
    k : int
        Number of top anomalies to consider for Precision@k
    
    Returns:
    --------
    dict: Evaluation metrics
    """
    # Get top k predictions
    top_k_indices = np.argsort(scores)[-k:]
    y_pred_top = np.zeros(len(y_true))
    y_pred_top[top_k_indices] = 1
    
    # Compute metrics
    precision_k = precision_score(y_true, y_pred_top)
    recall_k = recall_score(y_true, y_pred_top)
    f1_k = f1_score(y_true, y_pred_top)
    
    auc_roc = roc_auc_score(y_true, scores)
    auc_pr = average_precision_score(y_true, scores)
    
    return {
        'precision@100': precision_k,
        'recall@100': recall_k,
        'f1@100': f1_k,
        'auc_roc': auc_roc,
        'auc_pr': auc_pr
    }

def normalise_scores(scores):
    """Normalise scores to [0, 1] range."""
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-10)

print("✅ Evaluation functions defined!")

✅ Evaluation functions defined!


## 4. Train Models

### 4.1 Isolation Forest

Isolation Forest isolates anomalies by randomly selecting features and split values. Anomalies require fewer splits to isolate.

In [9]:
print("=" * 60)
print("Training Isolation Forest...")
print("=" * 60)

# Train Isolation Forest
if_model = IsolationForest(
    contamination=0.05,
    random_state=42,
    n_estimators=100,
    max_samples='auto'
)

if_model.fit(X_train)
if_scores = -if_model.score_samples(X_test)  # Higher = more anomalous

# Normalise scores
if_scores_norm = normalise_scores(if_scores)

print(f"Isolation Forest trained!")
print(f"Score range: [{if_scores_norm.min():.4f}, {if_scores_norm.max():.4f}]")

Training Isolation Forest...
Isolation Forest trained!
Score range: [0.0000, 1.0000]
